In [ ]:
import requests
import json
import os
import cv2


API_KEY = 'renbenit'
API_ENDPOINT = 'https://api.europeana.eu/record/v2/search.json'
QUERY_PARAMS = {
    'wskey': API_KEY,
    'query': 'provider_aggregation_edm_isShownBy:*mp4*',
    'qf': ['PROVIDER:"The European Film Gateway"', 'TYPE:VIDEO'],
    'reusability': 'open',
    'rows': 10,  
    'profile': 'rich',
    'start': 1
}

OUTPUT_DIR = 'europeana_videos'
os.makedirs(OUTPUT_DIR, exist_ok=True)

def download_video(url, filename):
    print(f"🔹 Downloading: {url} -> {filename}")
    response = requests.get(url, stream=True)
    if response.status_code == 200:
        with open(filename, 'wb') as f:
            for chunk in response.iter_content(chunk_size=1024):
                f.write(chunk)

def save_metadata(metadata, filename):
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(metadata, f, ensure_ascii=False, indent=4)
        

def main():
    start = 1
    while True:
        QUERY_PARAMS['start'] = start
        response = requests.get(API_ENDPOINT, params=QUERY_PARAMS)
        data = response.json()

        for item in data['items']:
            if 'edmIsShownBy' in item:
                video_url = item['edmIsShownBy']
                if isinstance(video_url, list):
                    video_url = video_url[0]
                title = item.get('title', ['untitled'])[0].replace('/', '_')
                video_filename = os.path.join(OUTPUT_DIR, f'{title}.mp4')
                metadata_filename = os.path.join(OUTPUT_DIR, f'{title}.json')
                download_video(video_url, video_filename)
                save_metadata(item, metadata_filename)
            else:
                print(f"No video URL found for item: {item.get('title', 'Unknown Title')}")

        start += len(data['items'])

if __name__ == '__main__':
    main()
